# Tangential Atlas Workflow

Manual atlas placement and lightweight inspection for tangential-volume outputs.

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from iss_preprocess.pipeline import (
    SliceResidual,
    StackPose,
    build_slice_plane_spec,
    build_tangential_atlas_context,
    load_atlas,
    load_tangential_atlas_state,
    pose_to_matrix,
    render_annotation_plane,
    render_reference_plane,
    write_qc_report,
    register_spots_to_tangential_atlas,
    write_tangential_atlas_rasters,
)
from iss_preprocess.vis.tangential_atlas_napari import review_tangential_atlas_napari
from iss_preprocess.io.load import get_processed_path

## Parameters

Set these before running the workflow. The section-thickness fallback for this atlas workflow is 20 um.

In [6]:
mouse_path = get_processed_path("essenbd_projdev/BRAC11398.3c/chamber_01").parent
stack_path = None  # or mouse_path / "tangential_volume" / "unregistered_slices.npz"
transforms_path = None  # or mouse_path / "tangential_volume" / "global_slice_transforms.npz"
state_path = None  # defaults to mouse_path / "tangential_atlas" / "tangential_atlas_state.json"

atlas_name = "allen_mouse_10um"
overview_pixel_size_um = None  # prefer metadata; set explicitly if needed
section_thickness_um = 20.0


## Load And Validate Context

In [7]:
context = build_tangential_atlas_context(
    mouse_path,
    stack_path=stack_path,
    transforms_path=transforms_path,
    state_path=state_path,
    overview_pixel_size_um=overview_pixel_size_um,
    section_thickness_um=section_thickness_um,
)

print(f"kept slices: {context.geometry.slice_numbers.tolist()}")
print(f"overview pixel size: {context.metadata.overview_pixel_size_um} um ({context.metadata.overview_pixel_size_source})")
print(f"section thickness: {context.metadata.section_thickness_um} um ({context.metadata.section_thickness_source})")
print(f"z source: {context.metadata.z_source}")
for issue in context.issues:
    print(f"{issue.severity}: {issue.code}: {issue.message}")


kept slices: [4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 59, 60, 61, 63, 64, 65, 66, 67, 68, 69, 70, 72, 74, 76, 78, 79, 83, 84, 89, 90, 91, 92, 94, 98, 99, 100, 101, 102, 103, 104, 105, 106, 108]
overview pixel size: 1.8504 um (manifest.overview_pixel_size_um)
section thickness: 20.0 um (explicit)
z source: slice_number_spacing


## Open Manual Reviewer

Move the atlas in napari and press the save button in the reviewer.

In [ ]:
viewer = review_tangential_atlas_napari(
    mouse_path,
    state_path=state_path,
    atlas_name=atlas_name,
    overview_pixel_size_um=overview_pixel_size_um,
    section_thickness_um=section_thickness_um,
)


## Reload Saved State

In [ ]:
if state_path is None:
    state_path = mouse_path / "tangential_atlas" / "tangential_atlas_state.json"

state = load_tangential_atlas_state(mouse_path, state_path=state_path)
state.keys()


## Render A Few Atlas Planes

In [ ]:
atlas = load_atlas(state.get("atlas_name", atlas_name))
pose_dict = state["pose"]
pose = StackPose(
    atlas_from_tangential=np.asarray(pose_dict["atlas_from_tangential_4x4"], dtype=float),
    yaw_deg=float(pose_dict.get("yaw_deg", 0.0)),
    pitch_deg=float(pose_dict.get("pitch_deg", 0.0)),
    roll_deg=float(pose_dict.get("roll_deg", 0.0)),
    depth_um=float(pose_dict.get("depth_um", 0.0)),
    tx_atlas_um=float(pose_dict.get("tx_atlas_um", 0.0)),
    ty_atlas_um=float(pose_dict.get("ty_atlas_um", 0.0)),
)

for slice_index in range(min(3, len(context.geometry.slice_numbers))):
    sn = int(context.geometry.slice_numbers[slice_index])
    prev = state.get("residuals", {}).get(str(sn), {})
    residual = SliceResidual(
        slice_number=sn,
        dx_um=float(prev.get("dx_um", 0.0)),
        dy_um=float(prev.get("dy_um", 0.0)),
        dtheta_deg=float(prev.get("dtheta_deg", 0.0)),
    )
    ref_spec = build_slice_plane_spec(context.geometry, pose, residual=residual, slice_index=slice_index)
    ann_spec = ref_spec.__class__(
        out_shape_yx=ref_spec.out_shape_yx,
        pixel_size_um=ref_spec.pixel_size_um,
        atlas_um_from_grid_4x4=ref_spec.atlas_um_from_grid_4x4,
        order=0,
        cval=0,
    )
    reference = render_reference_plane(atlas, ref_spec)
    annotation = render_annotation_plane(atlas, ann_spec)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(reference, cmap="gray")
    axes[0].set_title(f"slice {sn} reference")
    axes[1].imshow(annotation, cmap="tab20")
    axes[1].set_title(f"slice {sn} annotation")
    for ax in axes:
        ax.axis("off")
    plt.show()


## Write QC Report And Raster Sidecars

In [ ]:
qc = write_qc_report(mouse_path, state, atlas=atlas, context=context)
qc


In [ ]:
raster_path = write_tangential_atlas_rasters(
    mouse_path,
    state,
    context=context,
    atlas=atlas,
    include_coordinates=True,
    include_annotation=True,
)
raster_path


## Optional: Project Spots Into Atlas

Reads the mouse-level global spots table written by `register_spots_to_global_volume`,
projects every spot through the saved tangential atlas state, and writes
`tangential_atlas/<spots_prefix>_spots_atlas.pkl`. Bad-slice and unmapped rows are
preserved with `atlas_valid=False`.

In [ ]:
spots_atlas_path = register_spots_to_tangential_atlas(
    mouse_path,
    state_path=state_path,
    spots_path=None,
    spots_prefix="barcode_round",
    context=context,
    atlas=atlas,
    global_coordinate_unit="pixel",
    include_bad_slices=True,
)
spots_atlas_path